# CNN Image Classification and Streamlit Application
**Minor Project – Google Colab notebook**

Run every cell from top to bottom. This notebook trains a CNN on MNIST, creates the required single Python file, generates a PDF report with your actual training results, and packages the two submission files into one ZIP.

**Before submitting:** replace the student details in the next cell, run all cells, and check the generated ZIP.

In [1]:
# 1. Enter your details
STUDENT_NAME = "Your Name"
STUDENT_ID = "Your Student ID"
BATCH = "Your Batch"
SUBMISSION_DATE = "01 November 2026"


In [2]:
# 2. Install required packages
!pip -q install tensorflow streamlit reportlab scikit-learn
print("Required packages installed.")


Required packages installed.


In [3]:
# 3. Create the single Python project file
project_code = r'''"""
Minor Project: CNN Image Classification and Streamlit Application
Part A: MNIST handwritten digit classification using TensorFlow/Keras
Part B: Student Marks Calculator using Streamlit

Run CNN training in Colab:
    !python CNN_Streamlit_Project.py train

Run the Streamlit app:
    streamlit run CNN_Streamlit_Project.py
"""

import os
import sys
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

OUTPUT_DIR = "project_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = [str(i) for i in range(10)]


def build_cnn_model():
    """Create a compact CNN for 28x28 grayscale MNIST images."""
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


def train_cnn(epochs=5):
    """Load, preprocess, train and evaluate the MNIST CNN."""
    print("Loading MNIST dataset...")
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

    # Normalize pixel values from 0-255 to 0-1 and add channel dimension.
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0
    x_train = np.expand_dims(x_train, axis=-1)
    x_test = np.expand_dims(x_test, axis=-1)

    print("Training images:", x_train.shape)
    print("Testing images:", x_test.shape)
    print("Number of classes:", len(np.unique(y_train)))

    model = build_cnn_model()
    model.summary()

    history = model.fit(
        x_train, y_train,
        epochs=epochs,
        batch_size=128,
        validation_split=0.1,
        verbose=1
    )

    test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
    probabilities = model.predict(x_test, verbose=0)
    predictions = np.argmax(probabilities, axis=1)

    # Save model and numerical results for reproducibility.
    model.save(os.path.join(OUTPUT_DIR, "mnist_cnn_model.keras"))
    results = {
        "train_accuracy": [float(v) for v in history.history["accuracy"]],
        "val_accuracy": [float(v) for v in history.history["val_accuracy"]],
        "train_loss": [float(v) for v in history.history["loss"]],
        "val_loss": [float(v) for v in history.history["val_loss"]],
        "test_accuracy": float(test_accuracy),
        "test_loss": float(test_loss),
        "train_count": int(x_train.shape[0]),
        "test_count": int(x_test.shape[0]),
        "image_shape": list(x_train.shape[1:]),
        "class_count": int(len(np.unique(y_train)))
    }
    with open(os.path.join(OUTPUT_DIR, "results.json"), "w") as f:
        json.dump(results, f, indent=2)

    # Plot training/validation accuracy and loss.
    epochs_range = range(1, len(history.history["accuracy"]) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(epochs_range, history.history["accuracy"], marker="o", label="Training accuracy")
    plt.plot(epochs_range, history.history["val_accuracy"], marker="o", label="Validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training and Validation Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "accuracy.png"), dpi=150)
    plt.close()

    plt.figure(figsize=(7, 4))
    plt.plot(epochs_range, history.history["loss"], marker="o", label="Training loss")
    plt.plot(epochs_range, history.history["val_loss"], marker="o", label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "loss.png"), dpi=150)
    plt.close()

    # Save a grid showing actual and predicted labels.
    plt.figure(figsize=(10, 6))
    for i in range(12):
        plt.subplot(3, 4, i + 1)
        plt.imshow(x_test[i].squeeze(), cmap="gray")
        plt.title(f"Actual: {y_test[i]} | Pred: {predictions[i]}")
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "predictions.png"), dpi=150)
    plt.close()

    # Save a few correct and incorrect prediction examples.
    correct = np.where(predictions == y_test)[0][:5]
    incorrect = np.where(predictions != y_test)[0][:5]
    for label, indices in [("correct", correct), ("incorrect", incorrect)]:
        plt.figure(figsize=(10, 2.5))
        for j, idx in enumerate(indices):
            plt.subplot(1, max(1, len(indices)), j + 1)
            plt.imshow(x_test[idx].squeeze(), cmap="gray")
            plt.title(f"True {y_test[idx]}\nPred {predictions[idx]}")
            plt.axis("off")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f"{label}_examples.png"), dpi=150)
        plt.close()

    print(f"Test accuracy: {test_accuracy:.4f}")
    print(f"Test loss: {test_loss:.4f}")
    print("CNN training complete. Outputs saved in:", OUTPUT_DIR)
    return results


# ---------------------- PART B: STREAMLIT APPLICATION ----------------------

def run_streamlit_app():
    """Display a simple Student Marks Calculator in Streamlit."""
    import streamlit as st

    st.set_page_config(page_title="Student Marks Calculator", page_icon="📘", layout="centered")
    st.title("📘 Student Marks Calculator")
    st.write("Enter a student's marks to calculate total, percentage and result.")

    name = st.text_input("Student name")
    student_id = st.text_input("Student ID")
    st.subheader("Enter marks (0 to 100)")
    subject_count = st.number_input("Number of subjects", min_value=1, max_value=10, value=5, step=1)

    marks = []
    for i in range(int(subject_count)):
        mark = st.number_input(
            f"Subject {i + 1} marks",
            min_value=0.0, max_value=100.0, value=0.0, step=1.0,
            key=f"subject_{i}"
        )
        marks.append(mark)

    if st.button("Calculate Result", type="primary"):
        if not name.strip() or not student_id.strip():
            st.error("Please enter both student name and student ID.")
        elif len(marks) == 0:
            st.error("Please enter marks for at least one subject.")
        else:
            total = sum(marks)
            maximum = len(marks) * 100
            percentage = (total / maximum) * 100
            failed = any(m < 35 for m in marks)

            st.success(f"Result calculated for {name.strip()}")
            st.write(f"**Student ID:** {student_id.strip()}")
            st.metric("Total marks", f"{total:.1f} / {maximum:.1f}")
            st.metric("Percentage", f"{percentage:.2f}%")
            if failed:
                st.error("Overall result: FAIL (at least one subject is below 35).")
            else:
                st.success("Overall result: PASS")
                if percentage >= 90:
                    grade = "A+"
                elif percentage >= 80:
                    grade = "A"
                elif percentage >= 70:
                    grade = "B"
                elif percentage >= 60:
                    grade = "C"
                elif percentage >= 50:
                    grade = "D"
                else:
                    grade = "E"
                st.write(f"**Grade:** {grade}")
            st.bar_chart({"Marks": marks})


if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1].lower() == "train":
        train_cnn()
    else:
        run_streamlit_app()
'''
with open("CNN_Streamlit_Project.py", "w", encoding="utf-8") as f:
    f.write(project_code)
print("Created CNN_Streamlit_Project.py")


Created CNN_Streamlit_Project.py


In [ ]:
# 4. Train the CNN model (this may take several minutes)
!python CNN_Streamlit_Project.py train


I0000 00:00:1793526025.600453    1117 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793526025.674359    1117 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793526030.896417    1117 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Loading MNIST dataset...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training images: (60000, 28, 28, 1)
Testing images: (10000, 28, 28, 1)
Number of classes: 10
E0000 00:00:1793526034.924124    1117 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output S

In [ ]:
# 5. Display training graphs and sample predictions
from IPython.display import display, Image
display(Image("project_outputs/accuracy.png"))
display(Image("project_outputs/loss.png"))
display(Image("project_outputs/predictions.png"))


In [ ]:
# 6. Generate the PDF report using your actual model results
import json, os
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER

with open("project_outputs/results.json") as f:
    results = json.load(f)

pdf_path = "CNN_Project_Report.pdf"
doc = SimpleDocTemplate(pdf_path, pagesize=A4, rightMargin=42, leftMargin=42, topMargin=42, bottomMargin=42)
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="CoverTitle", parent=styles["Title"], alignment=TA_CENTER, spaceAfter=18))
story = []

story.append(Spacer(1, 0.7*inch))
story.append(Paragraph("MINOR PROJECT REPORT", styles["CoverTitle"]))
story.append(Paragraph("CNN Image Classification and Streamlit Application", styles["Heading1"]))
story.append(Spacer(1, 0.3*inch))
story.append(Paragraph(f"<b>Student Name:</b> {STUDENT_NAME}", styles["Normal"]))
story.append(Paragraph(f"<b>Student ID:</b> {STUDENT_ID}", styles["Normal"]))
story.append(Paragraph(f"<b>Batch:</b> {BATCH}", styles["Normal"]))
story.append(Paragraph(f"<b>Date:</b> {SUBMISSION_DATE}", styles["Normal"]))
story.append(Paragraph("<b>Dataset:</b> MNIST handwritten digits", styles["Normal"]))
story.append(Paragraph("<b>Streamlit App:</b> Student Marks Calculator", styles["Normal"]))
story.append(Spacer(1, 0.4*inch))
story.append(Paragraph("Submitted as part of the Minor Project assignment.", styles["BodyText"]))
story.append(PageBreak())

story.append(Paragraph("1. Objective", styles["Heading1"]))
story.append(Paragraph("The objective is to build a Convolutional Neural Network (CNN) that recognizes handwritten digits from 0 to 9. A second objective is to create a simple Streamlit application that accepts student marks and calculates the total, percentage, pass/fail status and grade.", styles["BodyText"]))
story.append(Spacer(1, 10))
story.append(Paragraph("2. Dataset and CNN Workflow", styles["Heading1"]))
story.append(Paragraph(f"The MNIST dataset contains {results['train_count']:,} training images and {results['test_count']:,} test images. Each image has shape {results['image_shape'][0]} × {results['image_shape'][1]} pixels with one grayscale channel. There are {results['class_count']} classes, representing digits 0 through 9.", styles["BodyText"]))
story.append(Paragraph("Preprocessing: pixel values were converted from the range 0–255 to 0–1. A channel dimension was added so the images could be passed to the CNN. The training data was split to use 10% for validation.", styles["BodyText"]))
story.append(Paragraph("Model: two convolution and max-pooling stages extract image features. The features are flattened and passed through a dense layer and dropout layer. The final softmax layer predicts probabilities for the ten digit classes. The model uses the Adam optimizer and sparse categorical cross-entropy loss.", styles["BodyText"]))
story.append(Spacer(1, 8))
story.append(Paragraph("Model summary: the model consists of convolution, pooling, flattening, dense, dropout and output layers. The output layer contains 10 neurons, one for each digit.", styles["BodyText"]))
story.append(PageBreak())

story.append(Paragraph("3. Training and Evaluation Results", styles["Heading1"]))
story.append(Paragraph(f"After {len(results['train_accuracy'])} training epochs, the test accuracy was <b>{results['test_accuracy']*100:.2f}%</b> and the test loss was <b>{results['test_loss']:.4f}</b>. Final training accuracy was {results['train_accuracy'][-1]*100:.2f}% and validation accuracy was {results['val_accuracy'][-1]*100:.2f}%.", styles["BodyText"]))
story.append(Spacer(1, 8))
story.append(RLImage("project_outputs/accuracy.png", width=6.1*inch, height=3.45*inch))
story.append(Spacer(1, 8))
story.append(RLImage("project_outputs/loss.png", width=6.1*inch, height=3.45*inch))
story.append(PageBreak())

story.append(Paragraph("4. Sample Predictions", styles["Heading1"]))
story.append(Paragraph("The following examples compare the actual digit with the CNN prediction. A correct prediction has matching labels; an incorrect prediction has different labels. Handwriting variation, image shape and similar-looking digits can cause mistakes.", styles["BodyText"]))
story.append(Spacer(1, 8))
story.append(RLImage("project_outputs/predictions.png", width=6.2*inch, height=3.7*inch))
story.append(Spacer(1, 8))
story.append(Paragraph("Correct prediction examples", styles["Heading2"]))
story.append(RLImage("project_outputs/correct_examples.png", width=6.0*inch, height=1.5*inch))
story.append(Paragraph("Incorrect prediction examples", styles["Heading2"]))
story.append(RLImage("project_outputs/incorrect_examples.png", width=6.0*inch, height=1.5*inch))
story.append(PageBreak())

story.append(Paragraph("5. Streamlit Application", styles["Heading1"]))
story.append(Paragraph("The Streamlit application is a Student Marks Calculator. The user enters a student name, student ID, number of subjects and marks for each subject. After clicking Calculate Result, the application displays total marks, percentage, pass/fail status and grade. It also displays a bar chart of subject marks.", styles["BodyText"]))
story.append(Paragraph("Input validation: the application checks that the student name and ID are not blank. Marks are restricted to the range 0–100, and the number of subjects is restricted to 1–10. A student fails if any subject mark is below 35.", styles["BodyText"]))
story.append(Paragraph("To run the application, install the requirements and execute: <b>streamlit run CNN_Streamlit_Project.py</b>. The app opens in a browser.", styles["BodyText"]))
story.append(Spacer(1, 12))
story.append(Paragraph("6. Conclusion, Challenges and Improvements", styles["Heading1"]))
story.append(Paragraph(f"This project demonstrates the basic image-classification workflow using a CNN and the development of an interactive Streamlit application. The model achieved {results['test_accuracy']*100:.2f}% test accuracy on MNIST. Challenges include learning image preprocessing, selecting CNN layers and interpreting training/validation results. Future improvements could include data augmentation, hyperparameter tuning, a deeper CNN, and an image-upload interface for digit recognition.", styles["BodyText"]))
story.append(Spacer(1, 10))
story.append(Paragraph("References: MNIST dataset provided through TensorFlow/Keras; TensorFlow documentation; Streamlit documentation.", styles["BodyText"]))

doc.build(story)
print("Created:", pdf_path)
print("Report pages should be approximately 5; open the PDF and verify it before submission.")


In [ ]:
# 7. Package exactly the two required submission files into one ZIP
import zipfile, os
zip_name = "CNN_Minor_Project_Submission.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    z.write("CNN_Streamlit_Project.py")
    z.write("CNN_Project_Report.pdf")
print("Created:", zip_name)
print("ZIP contents:")
with zipfile.ZipFile(zip_name) as z:
    for name in z.namelist():
        print("-", name)


## 8. Download and run the Streamlit app

Download `CNN_Minor_Project_Submission.zip` from the Colab Files panel (folder icon → three dots → Download). Upload this single ZIP to your LMS.

To test the Streamlit app in Colab, run the following in a new code cell. If the public tunnel does not open, you can download the `.py` file and run it on your computer after installing the listed packages.

```python
!pip -q install streamlit
!streamlit run CNN_Streamlit_Project.py --server.port 8501 &>/content/streamlit.log &
!npx localtunnel --port 8501
```

**Important checks before submission**
- Replace the student details at the top before running all cells.
- Run all cells from top to bottom without skipping the CNN training cell.
- Open the PDF and check your name, results, graphs and predictions.
- Test the Streamlit app and take a screenshot of its working output if your lecturer expects app screenshots. The report currently describes the app but does not include a live app screenshot.
- The ZIP contains only the required `.py` and `.pdf` files.